# Responses API
Responses API는 OpenAI의 최신 API 흐름이다.
기존 Chat Completions API처럼 텍스트를 생성할 수 있고, 구조화 출력, 도구 출력, 멀티턴 대화 같은 기능을 더 일관 된 방식으로 다룰 수 있다.

In [ ]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini


## 기본 호출
- 기존 Chat Completions의 system message는 instructions로 옮겨간다.
- user message(사용자의 요청)은 input으로 옮겨간다.
- 응답 객체의 output_text 로 접근하면 응답 텍스트를 확인할 수 있다.

In [ ]:
response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급 개발자에게 쉽게 설명하는 AI 강사이다.",
    input="Responses API의 핵심 특징을 3개 bullet으로 설명해줘.",
    temperature=0.3
)

print(response.output_text)

Responses API의 핵심 특징을 초급 개발자도 이해하기 쉽게 3가지로 정리해볼게요!

- **자동 응답 생성**: 사용자의 질문이나 요청에 대해 자동으로 적절한 답변을 만들어줘요. 예를 들어, 챗봇이나 고객 지원에 활용할 수 있어요.
- **다양한 입력 지원**: 텍스트뿐만 아니라 이미지 같은 다양한 형태의 입력도 받아서 처리할 수 있어요.
- **맞춤형 설정 가능**: 응답의 스타일이나 길이 등을 개발자가 원하는 대로 조절할 수 있어서, 서비스에 딱 맞는 답변을 만들 수 있어요.

필요하면 더 자세히 설명해줄게요!


## previous_response_id로 멀티턴 대화 만들기

In [ ]:
first = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급자에게 비유를 잘 드는 AI 강사다.",
    input="RAG를 한 문장으로 설명해줘"
)

print(first.output_text)
print("response id:", first.id)

RAG는 외부 문서에서 정보를 찾아와 답변을 생성하는 똑똑한 AI 기술이에요.
response id: resp_007a46ed3cbc59930069fbe5403ff081979055089e47b30d8c


In [ ]:
second = client.responses.create(
    model=DEFAULT_MODEL,
    previous_response_id=first.id,
    input="방금 설명을 도서관 비유로 바꿔줘."
)

print(second.output_text)

RAG는 도서관 사서가 필요한 책을 찾아와서 질문에 답해주는 똑똑한 시스템이에요.


## 반복 입력으로 대화 이어가기

In [ ]:
print("종료하려면 exit를 입력하세요.")

previous_response_id = None

while True:
    user_input = input("User: ")

    if user_input.strip().lower() == 'exit':
        print("채팅을 종료합니다.")
        break

    # 첫 요청에는 previous_response_id가 없으므로 두 번째 요청부터 전달한다
    request_params = {
        "model" : DEFAULT_MODEL,
        "instructions" : "너는 Python 수업을 돕는 AI 튜터이다. 답변은 3문장 이내로 한다.",
        "input" : user_input,
        "temperature" : 0.4
    }

    if previous_response_id is not None:
        request_params['previous_response_id'] = previous_response_id

    response = client.responses.create(**request_params)

    print("Assistant : ", response.output_text)

    # 다음 대화를 위해 이번 응답 id를 저장
    previous_response_id = response.id

종료하려면 exit를 입력하세요.
Assistant :  기본 문법부터 차근차근 배우고, 작은 프로젝트나 문제를 풀면서 실습하는 것이 좋아요. 모르는 부분은 구글 검색이나 온라인 강의를 참고하고, 꾸준히 코드를 작성하는 습관을 들이세요. 질문이 있으면 언제든 물어봐도 돼요!
Assistant :  파이썬은 문법이 간단하고 배우기 쉬워 초보자에게 적합해요. 또한 데이터 분석, 인공지능, 웹 개발 등 다양한 분야에서 널리 사용돼 실무 활용도가 높습니다. 덕분에 취업과 프로젝트에 큰 도움이 돼요.
Assistant :  파이썬은 배우기 쉬운 문법과 다양한 활용 분야 덕분에 좋은 프로그래밍 언어입니다. 공부할 때는 기본 문법부터 시작해 작은 프로젝트로 실습하며 꾸준히 연습하는 게 중요해요. 궁금한 점이 있으면 언제든 질문하세요!
채팅을 종료합니다.
